# Render every Phase-5 blade — grouped into PASSED vs FAILED

**Fresh Colab session — reads your Drive, re-runs no phase.** Loads the design *numbers* Phase 4
saved (`checkpoint.npz`), reads which designs Phase 5 verified (`verification.json`), rebuilds
**every** blade's 3D CAD solid with CadQuery, and renders them in two groups: the **valid** ones
(clean geometry, green) and the **suspect** ones (self-intersecting / failed, red). CadQuery only
(no gmsh/SU2). Rotate any plot by dragging.

## 1. Repo + deps + mount Drive

In [ ]:
import importlib.util, subprocess, sys
from pathlib import Path

IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    REPO = Path("/content/fan-optimization")
    if not REPO.exists():
        subprocess.run(["git","clone","-b","main","https://github.com/clingergab/fan-optimization.git",str(REPO)], check=True)
    else:
        subprocess.run(["git","-C",str(REPO),"pull","origin","main"], check=True)
    subprocess.run([sys.executable,"-m","pip","install","-q","cadquery","matplotlib"], check=True)
    from google.colab import drive; drive.mount("/content/drive")
else:
    REPO = Path.cwd()
    while REPO != REPO.parent and not (REPO/"pyproject.toml").exists(): REPO = REPO.parent
if str(REPO/"src") not in sys.path: sys.path.insert(0, str(REPO/"src"))
print("repo:", REPO, "| colab:", IN_COLAB)

## 2. Load the data + split passed vs failed

`checkpoint.npz` → every design's 35 numbers (`x`). `verification.json`'s `ranking.pairs` lists
every verified design with a `suspect` flag: `suspect=False` → **passed** (clean, real 3D J_fan),
`suspect=True` → **failed** (mesh self-intersection, or negative/reverse-thrust airflow).

In [ ]:
import numpy as np, json
CAMPAIGN = Path("/content/drive/MyDrive/fanopt/phase4_bo") if IN_COLAB else REPO/"data"/"phase4_bo_nb"
VERIFY   = Path("/content/drive/MyDrive/fanopt/phase5_verify") if IN_COLAB else REPO/"data"/"phase5_verify_nb"

x = np.load(CAMPAIGN/"checkpoint.npz")["x"]
pairs = json.loads((VERIFY/"verification.json").read_text())["ranking"]["pairs"]
passed = [p for p in pairs if not p["suspect"]]
failed = [p for p in pairs if p["suspect"]]
print(f"{len(pairs)} verified total  ->  {len(passed)} passed, {len(failed)} failed/suspect")

## 3. Rebuild + render helpers

`design numbers -> decode -> make_vunit_blade` (the CAD solid) `-> tessellate` (triangles) `->`
draw. `render_group` lays a list of designs out in a grid, one 3D view each, coloured by group.

In [ ]:
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import matplotlib.pyplot as plt
from fanopt.bo.codec import decode
from fanopt.geometry.generator import BladeDesignParams
from fanopt.geometry.fields import Layer2Params
from fanopt.geometry.primitives import Layer3Primitive
from fanopt.bo.inertia import NEUTRAL_LAYER4
from fanopt.geometry.assembly_cad import make_vunit_blade

def blade_triangles(vec, tol=0.001):
    params = BladeDesignParams(layer1=decode(vec), layer2=Layer2Params.all_inactive(),
                               layer3=Layer3Primitive.absent(), layer4=NEUTRAL_LAYER4)
    verts, tris = make_vunit_blade(params).val().tessellate(tol)
    V = np.array([[p.x, p.y, p.z] for p in verts])
    return V, [V[list(t)] for t in tris]

def draw(ax, title, vec, color):
    V, polys = blade_triangles(vec)
    ax.add_collection3d(Poly3DCollection(polys, alpha=0.55, facecolor=color,
                                         edgecolor="k", linewidth=0.05))
    for lo, hi, setl in zip(V.min(0), V.max(0), (ax.set_xlim, ax.set_ylim, ax.set_zlim)):
        setl(lo, hi)
    ax.set_box_aspect((1,1,1)); ax.set_title(title, fontsize=9); ax.view_init(elev=25, azim=-70)

def render_group(group, color, heading):
    if not group:
        print(f"(no {heading} designs)"); return
    cols = min(len(group), 3); rows = (len(group) + cols - 1) // cols
    fig = plt.figure(figsize=(5.2*cols, 4.6*rows))
    for i, p in enumerate(group):
        idx = int(p["name"].split("_i")[-1])
        j3 = p.get("j_fan_3d")
        sub = f"{p['name']}\nJ_fan_3D={j3:.2e}" if j3 is not None else f"{p['name']}\n(3D failed)"
        ax = fig.add_subplot(rows, cols, i+1, projection="3d")
        try: draw(ax, sub, x[idx], color)
        except Exception as e: ax.set_title(f"{p['name']}: CAD build failed\n{type(e).__name__}")
    fig.suptitle(heading, fontsize=14, y=1.0); plt.tight_layout(); plt.show()

## 4. PASSED designs — clean, buildable geometry (these are your real candidates)

In [ ]:
render_group(passed, "seagreen", f"PASSED ({len(passed)})  — valid, 3D-verified")

## 5. FAILED / SUSPECT designs — self-intersecting or reverse-thrust (rejected)

In [ ]:
render_group(failed, "crimson", f"FAILED / SUSPECT ({len(failed)})  — rejected")

## 6. Takeaway

Side by side: the **green** blades are clean sheets (buildable, real airflow); the **red** ones
fold through themselves or produce reverse thrust. The optimizer chased some of the red ones
because the cheap 2D slice couldn't see the 3D problem — which is exactly what the **V1.5
geometry-validity filter** (`docs/V2_backlog.md`) would catch up front.